In [ ]:
# Imports the 'os' module, which allows interaction with the operating system (e.g., environment variables, file paths).
import os 

# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'.
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' (i.e., the workspace bucket URL or path).
my_bucket


In [ ]:
# Load packages 
import pandas as pd 
from scipy.stats import false_discovery_control

# Iterate through chromosomes with available age association results
for chr in [2, 4, 15, 17]:
    
    # Read chromosome-specific SAIGE single-variant association results
    df = pd.read_csv(f"{my_bucket}/data/saige_step02/age_tests/EUR_biosample_age_single_assoc_chr{chr}", 
                     sep='\t')
    
    # Print the chromosome currently being processed
    print("Processing chromosome:", chr)
    
    # Retain variants with an allele frequency below 1%
    df = df[df['AF_Allele2'] < 0.01]
    
    # Check that no variants exceed the specified allele frequency threshold
    if (df['AF_Allele2'] > 0.01).any(): 
        break
    
    # Print the number of variants remaining after allele frequency filtering
    print(f"N variants:", len(df))
    
    # Calculate Benjamini-Hochberg adjusted p-values to account for multiple testing
    df['bh.p.value'] = false_discovery_control(df['p.value'], method='bh')
    
    # Identify variants with a positive effect size greater than 1 and nominal significance
    exclude = df[(df['BETA'] > 1) & (df['p.value'] < 0.05)]
    df = df[~(df['MarkerID'].isin(exclude['MarkerID']))]
    
    # Print the number of variants remaining after exclusion filtering
    print(f"N variants post filter:", len(df))
    
    # Export the remaining variant identifiers for use in downstream analyses
    df['MarkerID'].to_csv(f"{my_bucket}/data/saige_step01/biosample_markers/markers_chr{chr}", 
              sep='\t', 
              index=None,
              header=None)
    

# COMMAND 

In [ ]:
%%writefile biosample_age_restricted_analysis.sh

#!/bin/bash
set -o errexit
set -o nounset

# Loop through the populations
for pop in EUR; do

     # Loop through the phenotypes
    for pheno in biosample_age; do
   
        echo "Processing pop: ${pop} | phenotype: ${pheno} | chr: ${chrom}" 
        
        step2_SPAtests.R \
            --bedFile="${plink_bed}" \
            --bimFile="${plink_bim}" \
            --famFile="${plink_fam}" \
            --SAIGEOutputFile="${output}/${pop}_${pheno}_restricted_high_confidence_chr${chrom}" \
            --idstoIncludeFile="${snplist}" \
            --LOCO=FALSE \
            --AlleleOrder=alt-first \
            --minMAF=0 \
            --minMAC=0.5 \
            --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
            --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
            --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
            --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
            --groupFile="${vat}/HC-annotation-chr${chrom}.txt" \
            --annotation_in_groupTest=AlphaMissense-0.564,LOFTEE-HC:SpliceAI-0.8,LOFTEE-HC:SpliceAI-0.8:AlphaMissense-0.564 \
            --maxMAF_in_groupTest=0.0001,0.001,0.01 \
            --is_output_markerList_in_groupTest=TRUE \
            --is_fastTest=TRUE

    done
done


In [ ]:
# Save script to workspace bucket 
!gsutil cp biosample_age_restricted_analysis.sh {my_bucket}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gsutil cat {my_bucket}/data/scripts/biosample_age_restricted_analysis.sh

# COMMAND SUBMSSION 

In [ ]:
%%bash --out saige_gene_step02

# Loop through each chromosome 
for chromo in 2 4 15 17; do

  # Source the dsub configuration file
  source ~/aou_dsub.bash 

  # Submit a dsub command 
  aou_dsub \
    --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
    --boot-disk-size 10 \
    --disk-size 20 \
    --min-ram 4 \
    --logging "${WORKSPACE_BUCKET}/data/logging" \
    --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
    --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
    --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
    --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
    --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
    --input snplist="${WORKSPACE_BUCKET}/data/saige_step01/biosample_markers/markers_chr${chromo}" \
    --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/EUR_HC_annotation/" \
    --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/age_tests/" \
    --env chrom=${chromo} \
    --script "${WORKSPACE_BUCKET}/data/scripts/biosample_age_restricted_analysis.sh"

done